# Master End-to-End Runner

Runs **Phase 0 → Phase 5** notebooks in order and **fails fast** on errors or DQ gate failures.

**Run this notebook from the project root** (`Hospital project/`).

Power BI Desktop refresh/build remains manual (see `powerbi/BUILD_INSTRUCTIONS.md`).


## 1. Setup

Import notebook runner and confirm we are in the project root.


In [1]:
from __future__ import annotations
from pathlib import Path
import nbformat
from nbclient import NotebookClient

ROOT = Path(".").resolve()
if not (ROOT / "datafile.txt").exists():
    raise FileNotFoundError("Run master.ipynb from the project root")

print("ROOT:", ROOT)


ModuleNotFoundError: No module named 'nbformat'

## 2. Show data registry

All raw/clean/gold paths come from `datafile.txt`.


In [ ]:
print((ROOT / "datafile.txt").read_text(encoding="utf-8"))


## 3. Helper - run one phase notebook

Executes a notebook in-process and stops the pipeline on failure.


In [ ]:
def run_phase(phase_rel: str):
    path = ROOT / phase_rel
    print("\n" + "=" * 80)
    print("RUNNING", phase_rel)
    print("=" * 80)
    nb = nbformat.read(path, as_version=4)
    client = NotebookClient(
        nb,
        timeout=6000,
        kernel_name="hospital-dotvenv",
        resources={"metadata": {"path": str(ROOT)}},
    )
    try:
        client.execute()
    except Exception as e:
        print("FAILED:", phase_rel, e)
        nbformat.write(nb, path)
        raise SystemExit(f"Pipeline stopped at {phase_rel}: {e}")
    nbformat.write(nb, path)
    print("OK", phase_rel)


## 4. Run Phase 0 - Ingestion, lake, DQ, governance


In [ ]:
run_phase("notebooks/phase0_ingestion_lake_governance.ipynb")


## 5. Run Phase 1 - Modeling layer, SQL, marts


In [ ]:
run_phase("notebooks/phase1_modeling_marts_sql.ipynb")


## 6. Run Phase 2 - Stats, features, gold exports


In [ ]:
run_phase("notebooks/phase2_stats_features.ipynb")


## 7. Run Phase 3 - ML experiments and champion selection

This step can take a long time (full experiment matrix).


In [ ]:
run_phase("notebooks/phase3_ml_experiments.ipynb")


## 8. Run Phase 4 - Power BI certified exports


In [ ]:
run_phase("notebooks/phase4_powerbi_exports.ipynb")


## 9. Run Phase 5 - Clinician tool validation


In [ ]:
run_phase("notebooks/phase5_langgraph_app.ipynb")


## 10. Final registry and next steps


In [ ]:
print("\nPipeline complete. Final registry:")
print((ROOT / "datafile.txt").read_text(encoding="utf-8"))
print("Launch app: streamlit run app_streamlit.py")
print("Build Power BI using powerbi/BUILD_INSTRUCTIONS.md")
